# Spacing Statistics

## 1. Importing / Installing Packages

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None) # Show all columns when printing DataFrames

import numpy as np
import math

import datetime

from matplotlib import pyplot as plt
# Ensures that plots are displayed inline in Jupyter notebooks
%matplotlib inline
%config InlineBackend.figure_format = 'svg' # Configuring inline backend to use SVG format for figures

from dataclasses import dataclass
from enum import Enum, auto

from typing import Dict, Tuple, List, Union, Optional, ClassVar, Any, Literal, Iterable

from src.utils import DatabricksOdbcConnector, reorder_columns, compute_bg_rcat, read_excel_with_mapper, standardize_column_names
from src.well_data import WellDataLoader, GeoSurveyProcessor, DirectionalBenchNeighbors, WellSpacingCalculator, FloatingSectionWPS

## 2. Import Data

### 2.1 Importing Header

In [2]:
df_header_raw = read_excel_with_mapper(
    path=r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\05. Spacing Study\01. EF - 74 Ranch\header_ranch_74_EF.xlsx",
    sheet_name="header",
    col_map={"API14": "uwi14", "API12": "uwi12", "API10": "uwi10", "WellName":"well_name",
             "SpudDate":"spud_dt", "CompletionDate": "comp_dt",	"FirstProdDate":"first_prod_dt",
             "LastProdDate":"last_prod_dt"
             },
    dtype_map={"uwi14": str, "uwi12": str, "uwi10": str, "well_name": str},
    parse_dates=["SpudDate", "CompletionDate", "FirstProdDate", "LastProdDate"] # Parsing date columns during import because it’s handled inside pd.read_excel before we rename.
)

In [3]:
df_header_raw

,uwi14,uwi12,uwi10,well_name,Country,StateProvince,County,Lease,LeaseName,ENVOperator,ENVTicker,ENVWellStatus,Trajectory,Formation,spud_dt,comp_dt,first_prod_dt,last_prod_dt,BG_RCAT,ENV_Peer_Group,ENVCompanyType,ENV_Stock_Exchange,StateWellType,ENVWellType,ENVProducingMethod,ENVRegion,ENVBasin,ENVPlay,ENVSubPlay,ENVInterval,ENVIntervalSource,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Latitude,Longitude,Latitude_BH,Longitude_BH,TVD_FT,MD_FT,District,Field,Block,Abstract,Platform,Section,Township,Range,Section_Township_Range,Unit_Name,WellNumber,PlugDate,AvgFluidPerStage_BBL,FracRigOnsiteDate,FracRigReleaseDate,AvgProppantPerCluster_LBS,AvgProppantPerShot_LBS,AvgProppantPerStage_LBS,StimulatedStages,TotalClusters,Vintage,FirstProdQuarter,FirstProdMonth,CompletionTime_DAYS,PermitToSpud_DAYS,SpudToRigRelease_DAYS,SpudToCompletion_DAYS,SpudToSales_DAYS,SoakTime_DAYS,NumberOfStrings,UpperPerf_FT,LowerPerf_FT,PerfInterval_FT,LateralLength_FT,FracStages,AverageStageSpacing_FT,ProppantLoading_LBSPerGAL,ProppantIntensity_LBSPerFT,Proppant_LBS,TotalWaterPumped_GAL,WaterIntensity_GALPerFT,TotalFluidPumped_BBL,FluidIntensity_BBLPerFT,Bottom_Hole_Temp_DEGF,CasingPressure_PSI,FlowingTubingPressure_PSI,ShutInPressure_PSI,OilGravity_API,TotalProducingMonths,LastMonthLiquidsProduction_BBL,LastMonthGasProduction_MCF,LastMonthWaterProduction_BBL,TopOfZone_FT,BottomOfZone_FT,DensityPorosity_PCT,EffectivePorosity_PCT,ClayVolume_PCT,NonClayVolume_PCT,WaterSaturation_PCT,TotalOrganicCarbon_WTPCT,GasInitialRate,GasBFactor
0,42013343750000,420133437500,4201334375,GSH UNIT 10H,US,TX,ATASCOSA,15642,GSH UNIT,CRESCENT ENERGY COMPANY,CRGY,PRODUCING,HORIZONTAL,EAGLE FORD,2011-10-09 00:00:00,2012-01-03,2012-01-01,2025-09-01,NaN,MID CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD WEST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.677998,-98.404239,28.668345,-98.395768,10436.0,14834.0,1,EAGLEVILLE,NaN,963.0,NaN,23.0,NaN,NaN,NaN,GSH UNIT,10H,NaT,133427.0,NaN,NaN,NaN,NaN,6299342.0,NaN,NaN,2012.0,2012-Q1,2012 / 01,1.0,136.0,16.0,86.0,99.0,32.0,2.0,10510.0,14803.0,4293.0,4271.0,18.0,237.0,1.12,1467.0,6299342.0,5603934.0,1305.0,133427.0,31.0,NaN,NaN,2350.0,NaN,41.0,165.0,298.0,0.0,77.0,10173.0,10290.0,0.091,0.071,0.17,0.70,0.29,3.99,1.36,0.95
1,42013344800000,420133448000,4201334480,MIDDLE MCCOWEN 8H,US,TX,ATASCOSA,15638,MIDDLE MCCOWEN,CONOCOPHILLIPS,COP,PRODUCING,HORIZONTAL,EAGLE FORD,2012-05-22 00:00:00,2012-07-02,2012-07-01,2025-09-01,NaN,LARGE CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD EAST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.652433,-98.321220,28.665132,-98.333289,10891.0,16860.0,1,EAGLEVILLE,NaN,64.0,NaN,153.0,NaN,NaN,NaN,MIDDLE MCCOWEN,8H,NaT,8068.0,NaN,NaN,NaN,NaN,256083.0,NaN,NaN,2012.0,2012-Q3,2012 / 07,NaN,76.0,26.0,41.0,55.0,10.0,2.0,11199.0,16684.0,5485.0,5623.0,17.0,331.0,1.49,797.0,4371121.0,2940394.0,536.0,70009.0,13.0,NaN,2000.0,NaN,NaN,42.0,159.0,205.0,66.0,8.0,10721.0,10843.0,0.105,0.088,0.15,0.74,0.26,4.60,1.02,0.85
2,42013344820000,420133448200,4201334482,MIDDLE MCCOWEN 10H,US,TX,ATASCOSA,15638,MIDDLE MCCOWEN,CONOCOPHILLIPS,COP,PRODUCING,HORIZONTAL,EAGLE FORD,2012-06-11 00:00:00,2012-08-30,2012-08-01,2025-09-01,NaN,LARGE CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD EAST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.647486,-98.325239,28.631683,-98.310760,11119.0,18111.0,1,EAGLEVILLE,NaN,64.0,NaN,153.0,NaN,NaN,NaN,MIDDLE MCCOWEN,10H,NaT,54776.0,NaN,NaN,NaN,NaN,3642250.0,NaN,NaN,2012.0,2012-Q3,2012 / 08,NaN,95.0,21.0,80.0,66.0,18.0,2.0,11419.0,17991.0,6572.0,6755.0,19.0,356.0,1.58,554.0,3642250.0,2300592.0,350.0,54776.0,8.0,NaN,2200.0,2803.0,NaN,43.0,158.0,349.0,138.0,12.0,10934.0,11003.0,0.109,0.091,0.14,0.74,0.24,4.75,1.20,0.85
3,42013344880000,420133448800,4201334488,ET 1H,US,TX,ATASCOSA,16135,ET,MURPHY OIL,MUR,PRODUC

In [4]:
[col for col in df_header_raw.columns if 'ENV' in col]

['ENVOperator',
 'ENVTicker',
 'ENVWellStatus',
 'ENV_Peer_Group',
 'ENVCompanyType',
 'ENV_Stock_Exchange',
 'ENVWellType',
 'ENVProducingMethod',
 'ENVRegion',
 'ENVBasin',
 'ENVPlay',
 'ENVSubPlay',
 'ENVInterval',
 'ENVIntervalSource']

In [5]:
standardize_column_names(df_header_raw)

,uwi14,uwi12,uwi10,well_name,country,state_province,county,lease,lease_name,env_operator,env_ticker,env_well_status,trajectory,formation,spud_dt,comp_dt,first_prod_dt,last_prod_dt,b_g_r_c_a_t,env_peer_group,env_company_type,env_stock_exchange,state_well_type,env_well_type,env_producing_method,env_region,env_basin,env_play,env_sub_play,env_interval,env_interval_source,unnamed:_31,unnamed:_32,unnamed:_33,unnamed:_34,unnamed:_35,unnamed:_36,latitude,longitude,latitude_b_h,longitude_b_h,t_v_d_f_t,m_d_f_t,district,field,block,abstract,platform,section,township,range,section_township_range,unit_name,well_number,plug_date,avg_fluid_per_stage_b_b_l,frac_rig_onsite_date,frac_rig_release_date,avg_proppant_per_cluster_l_b_s,avg_proppant_per_shot_l_b_s,avg_proppant_per_stage_l_b_s,stimulated_stages,total_clusters,vintage,first_prod_quarter,first_prod_month,completion_time_d_a_y_s,permit_to_spud_d_a_y_s,spud_to_rig_release_d_a_y_s,spud_to_completion_d_a_y_s,spud_to_sales_d_a_y_s,soak_time_d_a_y_s,number_of_strings,upper_perf_f_t,lower_perf_f_t,perf_interval_f_t,lateral_length_f_t,frac_stages,average_stage_spacing_f_t,proppant_loading_l_b_s_per_g_a_l,proppant_intensity_l_b_s_per_f_t,proppant_l_b_s,total_water_pumped_g_a_l,water_intensity_g_a_l_per_f_t,total_fluid_pumped_b_b_l,fluid_intensity_b_b_l_per_f_t,bottom_hole_temp_d_e_g_f,casing_pressure_p_s_i,flowing_tubing_pressure_p_s_i,shut_in_pressure_p_s_i,oil_gravity_a_p_i,total_producing_months,last_month_liquids_production_b_b_l,last_month_gas_production_m_c_f,last_month_water_production_b_b_l,top_of_zone_f_t,bottom_of_zone_f_t,density_porosity_p_c_t,effective_porosity_p_c_t,clay_volume_p_c_t,non_clay_volume_p_c_t,water_saturation_p_c_t,total_organic_carbon_w_t_p_c_t,gas_initial_rate,gas_b_factor
0,42013343750000,420133437500,4201334375,GSH UNIT 10H,US,TX,ATASCOSA,15642,GSH UNIT,CRESCENT ENERGY COMPANY,CRGY,PRODUCING,HORIZONTAL,EAGLE FORD,2011-10-09 00:00:00,2012-01-03,2012-01-01,2025-09-01,NaN,MID CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD WEST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.677998,-98.404239,28.668345,-98.395768,10436.0,14834.0,1,EAGLEVILLE,NaN,963.0,NaN,23.0,NaN,NaN,NaN,GSH UNIT,10H,NaT,133427.0,NaN,NaN,NaN,NaN,6299342.0,NaN,NaN,2012.0,2012-Q1,2012 / 01,1.0,136.0,16.0,86.0,99.0,32.0,2.0,10510.0,14803.0,4293.0,4271.0,18.0,237.0,1.12,1467.0,6299342.0,5603934.0,1305.0,133427.0,31.0,NaN,NaN,2350.0,NaN,41.0,165.0,298.0,0.0,77.0,10173.0,10290.0,0.091,0.071,0.17,0.70,0.29,3.99,1.36,0.95
1,42013344800000,420133448000,4201334480,MIDDLE MCCOWEN 8H,US,TX,ATASCOSA,15638,MIDDLE MCCOWEN,CONOCOPHILLIPS,COP,PRODUCING,HORIZONTAL,EAGLE FORD,2012-05-22 00:00:00,2012-07-02,2012-07-01,2025-09-01,NaN,LARGE CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD EAST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.652433,-98.321220,28.665132,-98.333289,10891.0,16860.0,1,EAGLEVILLE,NaN,64.0,NaN,153.0,NaN,NaN,NaN,MIDDLE MCCOWEN,8H,NaT,8068.0,NaN,NaN,NaN,NaN,256083.0,NaN,NaN,2012.0,2012-Q3,2012 / 07,NaN,76.0,26.0,41.0,55.0,10.0,2.0,11199.0,16684.0,5485.0,5623.0,17.0,331.0,1.49,797.0,4371121.0,2940394.0,536.0,70009.0,13.0,NaN,2000.0,NaN,NaN,42.0,159.0,205.0,66.0,8.0,10721.0,10843.0,0.105,0.088,0.15,0.74,0.26,4.60,1.02,0.85
2,42013344820000,420133448200,4201334482,MIDDLE MCCOWEN 10H,US,TX,ATASCOSA,15638,MIDDLE MCCOWEN,CONOCOPHILLIPS,COP,PRODUCING,HORIZONTAL,EAGLE FORD,2012-06-11 00:00:00,2012-08-30,2012-08-01,2025-09-01,NaN,LARGE CAP,PUBLIC,NYSE,OIL_WELL,OIL,PUMPING,GULF COAST,WESTERN GULF,EAGLE FORD,EAGLE FORD EAST VOLATILE OIL,LOWER EAGLE FORD,ENV INTERPRETED,NaN,NaN,NaN,NaN,NaN,NaN,28.647486,-98.325239,28.631683,-98.310760,11119.0,18111.0,1,EAGLEVILLE,NaN,64.0,NaN,153.0,NaN,NaN,NaN,MIDDLE MCCOWEN,10H,NaT,54776.0,NaN,NaN,NaN,NaN,3642250.0,NaN,NaN,2012.0,2012-Q3,2012 / 08,NaN,95.0,21.0,80.0,66.0,18.0,2.0,11419.0,17991.0,6572.0,6755.0,19.0,356.0,1.58,554.0,3642250.0,2300592.0,350.0,5

In [6]:
[col for col in df_header_raw.columns if 'env' in col]

[]

In [4]:
compute_bg_rcat(df=df_header_raw,
                col_map = {
        "status": "ENVWellStatus",
        "last_prod": "last_prod_dt",
        "spud": "spud_dt",
        "comp": "comp_dt"
    })

0        1PDP
1        1PDP
2        1PDP
3        1PDP
4        1PDP
        ...  
2013    3PRMT
2014    3PRMT
2015     2DUC
2016     2DUC
2017    3PRMT
Length: 2018, dtype: object